In [ ]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==============================
# 2. TITLE
# ==============================
st.title("📊 YouTube Video Clustering")
st.write("Clustering videos based on views, likes, comments")

# ==============================
# 3. FILE UPLOAD
# ==============================
uploaded_file = st.file_uploader("Upload YouTube CSV file", type=["csv"])

if uploaded_file is not None:

    # ==============================
    # 4. LOAD DATA
    # ==============================
    df = pd.read_csv(uploaded_file)

    st.subheader("Dataset Preview")
    st.write(df.head())

    # ==============================
    # 5. SELECT REQUIRED COLUMNS
    # ==============================
    df = df[['views', 'likes', 'dislikes', 'comment_count', 'category_id']]
    df.rename(columns={'category_id': 'category'}, inplace=True)

    # ==============================
    # 6. CLEAN DATA
    # ==============================
    df.dropna(inplace=True)
    df = df.sample(5000, random_state=42)

    # ==============================
    # 7. FEATURES
    # ==============================
    X = df[['views', 'likes', 'dislikes', 'comment_count', 'category']]

    # ==============================
    # 8. SCALING
    # ==============================
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ==============================
    # 9. ELBOW METHOD
    # ==============================
    st.subheader("📉 Elbow Method")

    wcss = []
    for i in range(1, 10):
        model = KMeans(n_clusters=i, random_state=42)
        model.fit(X_scaled)
        wcss.append(model.inertia_)

    fig1, ax1 = plt.subplots()
    ax1.plot(range(1, 10), wcss)
    ax1.set_xlabel("Number of Clusters")
    ax1.set_ylabel("WCSS")
    ax1.set_title("Elbow Method")

    st.pyplot(fig1)

    # ==============================
    # 10. SELECT K
    # ==============================
    k = st.slider("Select number of clusters", 2, 10, 3)

    # ==============================
    # 11. APPLY KMEANS
    # ==============================
    kmeans = KMeans(n_clusters=k, random_state=42)
    df['cluster'] = kmeans.fit_predict(X_scaled)

    # ==============================
    # 12. PCA
    # ==============================
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    # ==============================
    # 13. VISUALIZATION
    # ==============================
    st.subheader("📊 Cluster Visualization")

    fig2, ax2 = plt.subplots()
    ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster'])
    ax2.set_xlabel("PCA 1")
    ax2.set_ylabel("PCA 2")
    ax2.set_title("Clusters")

    st.pyplot(fig2)

    # ==============================
    # 14. CLUSTER ANALYSIS
    # ==============================
    st.subheader("📈 Cluster Analysis")
    st.write(df.groupby('cluster').mean())